In [1]:
#Step 1 — Parse and clean input data
#Loops over results, extracts candidate metadata (cand_dict) and folding output (plot_info), normalizes formats 
#(handles list/None cases), and prepares required fields (notes, file paths, etc.).

#Step 2 — Compute and organize candidate parameters
#Builds derived quantities: converts RA/Dec using SkyCoord, selects best values for frequency (f0), dispersion measure (dm),
#SNR, and constructs paths to diagnostic plots and summary PDFs.

#Step 3 — Upload candidate to database
#Calls sd.add_candidate(...) to register each candidate with all computed parameters and file links, 
#then prints verification info (with optional final commit to save changes).

from sps_pipeline.candidate_viewer import CandidateViewerRegistrar
import sps_databases
import subprocess
from cfbm.bm_data import get_data
import os
import scipy
import numpy as np
from datetime import datetime, timedelta
from sps_databases import db_utils, db_api
from pathlib import Path
import astropy.units as u
from astropy.coordinates import SkyCoord
#here is the link git to the fct:https://github.com/chime-sps/champss_software/blob/main/champss/sps-pipeline/sps_pipeline/candidate_viewer.py#L215v

In [2]:
# Database configuration
db_config = {
    'host': 'sps-archiver1',
    'user': 'automation',
    'port': 3306,
    'password': '',#no password for automation user
    'database': 'champss'
}
db = db_utils.connect(name="champss_processing")

#We will create a separate folder in the multiday folds survey, called followup, or something along those lines.
#Maybe we create one new folder for the followup sources from each month.

sd = CandidateViewerRegistrar(
        survey="multiday",  #the project name under top-right corner of the website
        folder="single_day_1", #create a new folder
        db_config=db_config,
        survey_dir="/data/candidate_viewer/champss_candidate_viewer/surveys",  # path to the directory containing survey (project) config files
    )

In [3]:
results = np.load("/data/lkuenkel/run_mdf/all_out_combined.npy", allow_pickle=True)

#print(results.shape)
print(results[1])

[{'file': 'Multi_Pointing_Groups_f_1.820_DM_24.186_696bba5cdad56d5360048241', 'folder': '2026-01-07', 'result': '<faint>', 'date': 1768854315, 'modified_by': 'Rémi Tellier', 'info': '[]', 'id': 7693, 'rater_a': '', 'result_a': '', 'rater_b': '', 'result_b': '', 'additional_ratings': '', 'metadata': {'survey': 'dailycands', 'folder': '2026-01-07', 'file': 'Multi_Pointing_Groups_f_1.820_DM_24.186_696bba5cdad56d5360048241', 'input_file': '/mnt/beegfs-client/processed/mp_runs/daily_20260107/candidates/Multi_Pointing_Groups_f_1.820_DM_24.186_696bba5cdad56d5360048241.npz', 'candidate': '', 'telescope': 'chime', 'epoch_topo': '', 'epoch_bary': '', 't_sample': '', 'data_folded': '', 'data_avg': '', 'data_stdev': '', 'profile_bins': '', 'profile_avg': '', 'profile_stdev': '', 'reduce_chi_sqr': '', 'prob_noise': '17.741912841796875', 'best_dm': '24.18630601260553', 'p_topo': '', 'p_topo_d1': '', 'p_topo_d2': '', 'p_bary': '549.3895541852099', 'p_bary_d1': '0', 'p_bary_d2': '0', 'p_orb': '', 'asi

In [4]:
#Adding candidate to website
counter = 0
for row in results:
    counter += 1

    cand_dict = row[0]  # candidate metadata
    plot_info = row[1]  # could be dict, list of dicts, or None

    # If plot_info is a list, take the first dict(to avoid error when data is not there for f0)
    if isinstance(plot_info, list) and len(plot_info) > 0:
        plot_info = plot_info[0]
    elif plot_info is None:
        plot_info = {}

    md = cand_dict['metadata']
    notes = md.get('notes', {})

    #to get the pdf file(from /data/lkuenkel/run_mdf/get_summary_file.ipynb)
    mdf_plot = row[1]["path_to_plot"]
    mdf_plot_path = Path(mdf_plot)
    print(mdf_plot_path)
  
    print(md)

    fs = db.followup_sources.find_one({"path_to_candidates": md["input_file"]})
    summary_pdf = str(mdf_plot_path.parent / (fs["source_name"] + "_summary.pdf"))
    print(fs["source_name"])
    # Convert RA/Dec
    coord = SkyCoord(
        md.get('source_ra', '0:0:0'),
        md.get('source_dec', '0:0:0'),
        unit=(u.hourangle, u.deg)
    )

    ra_deg = coord.ra.deg
    dec_deg = coord.dec.deg

    # Assign variables safely
    f0_val = float(plot_info.get('f0', md.get('freq', 0)))  # from second column if available
    dm_val = float(md.get('best_dm', 0))
    snr_val = float(plot_info.get('SN', notes.get('fs_sigma', 0))) #Take SN if it exists, if not take fs_sigma
    input_file_val = md.get('input_file', '')
    fs_id_val = notes.get('fs_id', 'not_specified')
    fs_sigma_val = notes.get('fs_sigma', 'not_specified')
    fs_file_val = notes.get('fs_file', 'not_specified')

    # Extract paths if available
    phase_plot = plot_info.get('path_to_plot', None)

    # Call your function
    sd.add_candidate(
        candname=md.get('file', 'unknown'),
        ra=ra_deg,
        dec=dec_deg,
        f0=f0_val,
        dm=dm_val,
        snr=snr_val,
        input_file=input_file_val,
        fs_id=fs_id_val,
        fs_sigma=fs_sigma_val,
        fs_file=fs_file_val,
        phase_search_diagnostics=phase_plot,
        summary_pdf=summary_pdf,
        #**{k: v for k, v in md.items() if k not in ['file','source_ra','source_dec','freq','best_dm','input_file','notes']}
    )

    #Print verification
    print(f"Candidate #{counter}")
    print("RA:", ra_deg)
    print("Dec:", dec_deg)
    print("DM:", dm_val)
    print("f0:", f0_val)
    print("SNR",snr_val)
    print("Phase plot:", phase_plot)
    print("PDF path:", summary_pdf)
    print("-"*40) #just a line to separate candidate

#sd.commit()  # commit!!! to database and update survey config ,after the loop line 310 git

/mnt/beegfs-client/processed/archives/candidates/227.88_55.61/phase_search_22.26_1.36.png
{'survey': 'dailycands', 'folder': '2026-01-07', 'file': 'Multi_Pointing_Groups_f_1.364_DM_22.264_696bac92dad56d5360014de7', 'input_file': '/mnt/beegfs-client/processed/mp_runs/daily_20260107/candidates/Multi_Pointing_Groups_f_1.364_DM_22.264_696bac92dad56d5360014de7.npz', 'candidate': '', 'telescope': 'chime', 'epoch_topo': '', 'epoch_bary': '', 't_sample': '', 'data_folded': '', 'data_avg': '', 'data_stdev': '', 'profile_bins': '', 'profile_avg': '', 'profile_stdev': '', 'reduce_chi_sqr': '', 'prob_noise': '24.87929916381836', 'best_dm': '22.26354528356996', 'p_topo': '', 'p_topo_d1': '', 'p_topo_d2': '', 'p_bary': '733.0077518506666', 'p_bary_d1': '0', 'p_bary_d2': '0', 'p_orb': '', 'asin': '', 'eccentricity': '', 'w': '', 't_peri': '', 'header_size': '', 'data_size': '', 'data_type': '', 'notes': {'fs_id': '696c0128dad56d53600b899c', 'fs_sigma': 24.87929916381836, 'fs_file': '/mnt/beegfs-clien